# SGD vs Mini-Batch Gradient DescentComparing how a single-layer feedforward network learns under different gradient descent variants - full-batch, stochastic, and mini-batch.

## Batch, Stochastic, and Mini-Batch Gradient DescentAll three are the same core idea — update weights using the gradient of the loss — they differ only in **how much data** is used per update:- **Batch (Full) Gradient Descent** — computes the gradient using the *entire* dataset before each update. Stable, but slow per step and can get stuck in shallow local minima on non-convex problems.- **Stochastic Gradient Descent (SGD)** — updates weights using *one* sample at a time. Fast and noisy — the noise actually helps escape shallow local minima, but the loss curve is jittery.- **Mini-Batch Gradient Descent** — updates using a small batch (e.g. 8, 16, 32 samples). A practical middle ground: less noisy than SGD, faster than full-batch, and the standard choice in real deep learning (this is what Keras's `fit()` does by default with its `batch_size` argument).We'll implement a single-layer feedforward network (just one neuron layer, no hidden layer) from scratch and train it three ways on the same data to see the difference directly.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)# Simple binary classification dataset, linearly separable-ish with noisen_samples = 200X = np.random.randn(n_samples, 2)true_w = np.array([2.0, -3.0])true_b = 0.5y_prob = 1 / (1 + np.exp(-(X @ true_w + true_b)))y = (y_prob > 0.5).astype(float).reshape(-1, 1)plt.scatter(X[:,0], X[:,1], c=y.ravel(), cmap="bwr", alpha=0.6)plt.title("Dataset")plt.show()

In [ ]:
def sigmoid(z):    return 1 / (1 + np.exp(-z))def train_single_layer(X, y, batch_size, epochs=50, lr=0.1, seed=0):    rng = np.random.default_rng(seed)    n = X.shape[0]    w = rng.standard_normal((X.shape[1], 1)) * 0.1    b = np.zeros((1, 1))    losses = []    for epoch in range(epochs):        # shuffle each epoch        idx = rng.permutation(n)        X_shuffled, y_shuffled = X[idx], y[idx]        for start in range(0, n, batch_size):            end = start + batch_size            X_batch = X_shuffled[start:end]            y_batch = y_shuffled[start:end]            z = X_batch @ w + b            a = sigmoid(z)            error = a - y_batch            dw = X_batch.T @ error / len(X_batch)            db = np.mean(error, axis=0, keepdims=True)            w -= lr * dw            b -= lr * db        # loss over full dataset, for fair comparison across methods        a_full = sigmoid(X @ w + b)        epoch_loss = np.mean((y - a_full) ** 2)        losses.append(epoch_loss)    return w, b, losses

In [ ]:
# Batch GD: batch_size = full datasetw_batch, b_batch, losses_batch = train_single_layer(X, y, batch_size=len(X), epochs=50, lr=0.5)# SGD: batch_size = 1w_sgd, b_sgd, losses_sgd = train_single_layer(X, y, batch_size=1, epochs=50, lr=0.5)# Mini-batch: batch_size = 16w_mini, b_mini, losses_mini = train_single_layer(X, y, batch_size=16, epochs=50, lr=0.5)print("Final loss - Batch GD:", losses_batch[-1])print("Final loss - SGD:", losses_sgd[-1])print("Final loss - Mini-Batch:", losses_mini[-1])

In [ ]:
plt.figure(figsize=(10,6))plt.plot(losses_batch, label="Batch GD")plt.plot(losses_sgd, label="SGD")plt.plot(losses_mini, label="Mini-Batch (16)")plt.title("Loss over Epochs - Batch vs SGD vs Mini-Batch")plt.xlabel("Epoch")plt.ylabel("Loss (MSE)")plt.legend()plt.grid(True)plt.show()

## Observations- **Batch GD** converges smoothly but takes larger, more deliberate steps — one update per epoch.- **SGD** converges fast in terms of epochs but the path is noisier — visible as jitter in the loss curve — since each update reacts to a single sample.- **Mini-Batch** sits between the two: smoother than SGD, faster per-epoch progress than full Batch GD, since it makes multiple updates per epoch instead of one.

## Keras VerificationSame single-layer network, same three settings, using Keras's `batch_size` argument directly.

In [ ]:
import tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersdef build_and_train(batch_size, epochs=50):    model = keras.Sequential([        layers.Dense(1, activation='sigmoid', input_shape=(2,))    ])    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.5), loss='mse')    history = model.fit(X, y, batch_size=batch_size, epochs=epochs, verbose=0)    return history.history['loss']loss_batch_keras = build_and_train(batch_size=len(X))loss_sgd_keras = build_and_train(batch_size=1)loss_mini_keras = build_and_train(batch_size=16)plt.figure(figsize=(10,6))plt.plot(loss_batch_keras, label="Batch GD (Keras)")plt.plot(loss_sgd_keras, label="SGD (Keras)")plt.plot(loss_mini_keras, label="Mini-Batch (Keras)")plt.title("Keras: Batch vs SGD vs Mini-Batch")plt.xlabel("Epoch")plt.ylabel("Loss")plt.legend()plt.grid(True)plt.show()

## ConclusionBatch size controls the tradeoff between update stability and update frequency. Mini-batch gradient descent is the practical default in deep learning — it's what Keras uses whenever you pass a `batch_size` to `fit()`.